# Step 1 â€” Chunk Data
**Purpose:** Read every `.txt` file from `knowledge_base/pages/`, split by `[ SECTION ]` markers,
clean the text, attach metadata, and save all structured chunks to `knowledge_base/chunks.json`.

**Input:** `knowledge_base/pages/*.txt`  
**Output:** `knowledge_base/chunks.json`

> Run this notebook from the `backend/` directory or adjust the paths below.

## 1. Imports & Configuration

In [1]:
import os
import re
import json

# Paths are relative to the project root (one level above backend/)
_HERE       = os.path.dirname(os.path.abspath('__file__'))
_ROOT       = os.path.dirname(_HERE)  # project root

PAGES_DIR   = os.path.join(_ROOT, 'knowledge_base', 'pages')
OUTPUT_FILE = os.path.join(_ROOT, 'knowledge_base', 'chunks.json')

print(f'Pages directory : {os.path.abspath(PAGES_DIR)}')
print(f'Output file     : {os.path.abspath(OUTPUT_FILE)}')

Pages directory : D:\Users\Danish\Desktop\Projects\MANO-Website\knowledge_base\pages
Output file     : D:\Users\Danish\Desktop\Projects\MANO-Website\knowledge_base\chunks.json


## 2. List Available Page Files

In [2]:
page_files = sorted(f for f in os.listdir(PAGES_DIR) if f.endswith('.txt'))
print(f'Found {len(page_files)} page files:\n')
for f in page_files:
    print(f'  {f}')

Found 14 page files:

  about_us.txt
  careers.txt
  landing_page.txt
  projects.txt
  service_contract_management.txt
  service_cost_consultancy.txt
  service_ehs_auditing.txt
  service_epc.txt
  service_project_execution.txt
  service_project_management.txt
  service_project_planning.txt
  service_qa_qc_auditing.txt
  service_qs_billing.txt
  services_overview.txt


## 3. Core Parsing Function
For each file:
1. Read `page_name` and `url` from the header lines
2. Split on `===` separator lines to isolate `[ SECTION N ]` blocks
3. Strip the section label and clean whitespace
4. Create a structured chunk dict with metadata

In [3]:
def parse_pages_to_chunks(pages_dir: str, page_files: list) -> list:
    all_chunks = []
    chunk_id   = 0

    for filename in page_files:
        filepath = os.path.join(pages_dir, filename)

        with open(filepath, 'r', encoding='utf-8') as f:
            raw = f.read()

        # -- Step 1: Read page-level metadata from the header --
        page_name = 'Unknown'
        page_url  = ''
        for line in raw.splitlines()[:5]:
            if line.startswith('PAGE:'):
                page_name = line.replace('PAGE:', '').strip()
            elif line.startswith('URL:'):
                page_url = line.replace('URL:', '').strip()

        # -- Step 2: Split on '===' separator lines --
        raw_blocks = re.split(r'={3,}', raw)

        page_chunks_count = 0

        for block in raw_blocks:
            block = block.strip()

            # -- Step 3: Strip '[ SECTION N ]' label and skip small/header blocks --
            block = re.sub(r'^\[\s*SECTION\s*\d+\s*\]\s*', '', block).strip()
            if not block or block.startswith('PAGE:') or len(block) < 30:
                continue

            # -- Step 4: Clean text (strip lines, remove blanks) --
            lines         = [ln.strip() for ln in block.splitlines() if ln.strip()]
            clean_content = '\n'.join(lines)

            if len(clean_content) < 30:
                continue

            # -- Step 5: Build the structured chunk dict --
            all_chunks.append({
                'id':          f'chunk_{chunk_id:04d}',
                'page_name':   page_name,
                'url':         page_url,
                'source_file': filename,
                'section_num': page_chunks_count,
                'content':     clean_content,
            })
            chunk_id          += 1
            page_chunks_count += 1

        print(f'  [OK] {filename:45s}  ->  {page_chunks_count} chunks')

    return all_chunks

print('Function defined.')

Function defined.


## 4. Run the Parser

In [4]:
chunks = parse_pages_to_chunks(PAGES_DIR, page_files)
print(f'\nTotal chunks created: {len(chunks)}')

  [OK] about_us.txt                                   ->  7 chunks
  [OK] careers.txt                                    ->  8 chunks
  [OK] landing_page.txt                               ->  10 chunks
  [OK] projects.txt                                   ->  4 chunks
  [OK] service_contract_management.txt                ->  10 chunks
  [OK] service_cost_consultancy.txt                   ->  10 chunks
  [OK] service_ehs_auditing.txt                       ->  10 chunks
  [OK] service_epc.txt                                ->  13 chunks
  [OK] service_project_execution.txt                  ->  10 chunks
  [OK] service_project_management.txt                 ->  11 chunks
  [OK] service_project_planning.txt                   ->  11 chunks
  [OK] service_qa_qc_auditing.txt                     ->  10 chunks
  [OK] service_qs_billing.txt                         ->  10 chunks
  [OK] services_overview.txt                          ->  9 chunks

Total chunks created: 133


## 5. Preview a Sample Chunk

In [5]:
# Preview the first 3 chunks to verify structure
for chunk in chunks[:3]:
    print(json.dumps(chunk, indent=2, ensure_ascii=False))
    print('-' * 60)

{
  "id": "chunk_0000",
  "page_name": "ABOUT US",
  "url": "http://localhost:5173/pcpl/about-us",
  "source_file": "about_us.txt",
  "section_num": 0,
  "content": "DEDICATED TO EXCELLENCE\nWho\nWe Are\nWe are a team of dedicated professionals committed to delivering excellence in every project we undertake. From concept to completion, we ensure quality and precision.\nStart Your Project\nMeet the Team\n100+\nProjects Managed\nTimely Completion\n95%\n12+\nYEARS\n50+\nEXPERTS\n100%\nCOMMITMENT\nHIRING\nQUALIFIED"
}
------------------------------------------------------------
{
  "id": "chunk_0001",
  "page_name": "ABOUT US",
  "url": "http://localhost:5173/pcpl/about-us",
  "source_file": "about_us.txt",
  "section_num": 1,
  "content": "About The Company\nMANO Projects Pvt. Ltd. is a multi disciplinary Construction & Management firm established in September 2010 by Mr. Mugilan Muthaiah and Mrs. Amirthavalli Mugilan, having its head office in Mumbai.\nOur firm consists of young, qualif

## 6. Save All Chunks to `chunks.json`

In [6]:
os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_FILE)), exist_ok=True)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f'[SUCCESS] Saved {len(chunks)} chunks to:')
print(f'          {os.path.abspath(OUTPUT_FILE)}')
print('\nNext step -> run: backend/index_knowledge.py')

[SUCCESS] Saved 133 chunks to:
          D:\Users\Danish\Desktop\Projects\MANO-Website\knowledge_base\chunks.json

Next step -> run: backend/index_knowledge.py


## 7. Summary Statistics

In [7]:
from collections import Counter

page_counts = Counter(c['page_name'] for c in chunks)
print(f"{'Page':<45}  Chunks")
print('-' * 55)
for page, count in sorted(page_counts.items()):
    print(f'{page:<45}  {count}')
print('-' * 55)
print(f"{'TOTAL':<45}  {len(chunks)}")

Page                                           Chunks
-------------------------------------------------------
ABOUT US                                       7
CAREERS                                        8
LANDING PAGE                                   10
PROJECTS                                       4
SERVICE CONTRACT MANAGEMENT                    10
SERVICE COST CONSULTANCY                       10
SERVICE EHS AUDITING                           10
SERVICE EPC                                    13
SERVICE PROJECT EXECUTION                      10
SERVICE PROJECT MANAGEMENT                     11
SERVICE PROJECT PLANNING                       11
SERVICE QA QC AUDITING                         10
SERVICE QS BILLING                             10
SERVICES OVERVIEW                              9
-------------------------------------------------------
TOTAL                                          133
